# 🐻 Bear Detector — Google Colab Notebook

End-to-end pipeline: **detection → tracking → segmentation → evaluation**

| Step | Description |
|------|-------------|
| 1 | Setup: clone repo, install deps, check GPU |
| 2 | Load configuration |
| 3 | Train YOLOv8 detection model |
| 4 | Detect bears in a sample image |
| 5 | Detect + track bears in a video |
| 6 | Train and run segmentation |
| 7 | Evaluate: mAP, PR curve |
| 8 | Browse experiment logs |

> **GPU recommended.** Go to `Runtime → Change runtime type → T4 GPU` before running.

## 1 · Setup

In [ ]:
# Clone or update the repository and switch to the main branch
import os

REPO   = 'Bear-Detector'
BRANCH = 'main'

if os.path.isfile('pyproject.toml') or os.path.isdir('.git'):
    print('Already inside repo — fetching latest changes...')
    os.system(f'git fetch origin')
    os.system(f'git checkout {BRANCH}')
    os.system(f'git pull --ff-only')
elif os.path.isdir(REPO):
    print('Repo found locally — fetching latest changes...')
    os.system(f'git -C {REPO} fetch origin')
    os.system(f'git -C {REPO} checkout {BRANCH}')
    os.system(f'git -C {REPO} pull --ff-only')
    os.chdir(REPO)
else:
    os.system(f'git clone -b {BRANCH} https://github.com/danort92/Bear-Detector.git')
    os.chdir(REPO)

# Clear cached src.* modules so the freshly-pulled code is always used
import sys
for key in [k for k in sys.modules if k.startswith('src.')]:
    del sys.modules[key]

print(f'✅ Repository ready (branch: {BRANCH}, cwd: {os.getcwd()})')

In [ ]:
# Install dependencies (takes ~2 min on first run)
!pip install -q -r requirements.txt
print('✅ Dependencies installed')

In [ ]:
import sys, torch
sys.path.insert(0, '.')

from src.utils.device import get_device
from src.utils.seed import set_seed

DEVICE = get_device('auto')
set_seed(42)

print(f'🖥  Device  : {DEVICE}')
print(f'🔥 PyTorch : {torch.__version__}')
if DEVICE == 'cuda':
    print(f'🎮 GPU     : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2 · Load Configuration

In [ ]:
from src.utils.config import load_merged_config
import yaml

cfg = load_merged_config('config/default.yaml')

# ── Override anything you like here ──────────────────────────
cfg['training']['detection']['epochs']      = 20    # increase for better results
cfg['training']['detection']['batch_size']  = 16
cfg['experiment']['seed']                   = 42
cfg['mlflow']['enabled']                    = False  # set True to use MLflow UI
# ─────────────────────────────────────────────────────────────

print(yaml.dump(cfg, default_flow_style=False))

## 3 · Train YOLOv8 Detection Model

The dataset is the 1,172-image Roboflow bear-face dataset already included in the repo.
Training with `yolov8n` (nano) takes ~5–10 min on a T4 GPU for 20 epochs.

In [ ]:
import os
from src.training.train_detection import DetectionTrainer
from src.training.experiment_tracker import ExperimentTracker
from src.utils.seed import set_seed

set_seed(cfg['experiment']['seed'])

# Point Ultralytics' built-in MLflow callback to the same directory the UI reads from
MLFLOW_DIR = 'outputs/experiments/mlruns'
os.makedirs(MLFLOW_DIR, exist_ok=True)
os.environ['MLFLOW_TRACKING_URI'] = MLFLOW_DIR

tracker = ExperimentTracker(cfg)

with tracker.start_run() as run:
    run.log_params({
        'architecture' : cfg['model']['detection']['architecture'],
        'epochs'       : cfg['training']['detection']['epochs'],
        'lr'           : cfg['training']['detection']['learning_rate'],
        'batch_size'   : cfg['data']['detection']['batch_size'],
        'seed'         : cfg['experiment']['seed'],
    })
    trainer = DetectionTrainer(cfg)
    output  = trainer.train()

    BEST_PT = output['best_weights']
    print(f'\n✅ Training done. Best weights → {BEST_PT}')

    try:
        metrics = trainer.validate(BEST_PT)
        run.log_metrics(metrics)
        print(f'📊 Validation metrics: {metrics}')
    except Exception as exc:
        print(f'⚠️  Validation failed: {exc}')

## 4 · Detect Bears in Images

**Input options** (set `USE_UPLOAD = True` to activate the file picker):

| What you select | Result |
|-----------------|--------|
| One image (`.jpg`, `.png`, …) | Runs detection on that image |
| Multiple images (Ctrl/Cmd+click) | Runs detection on each image |
| A `.zip` of images or folders of images | Extracts and runs detection on everything inside |

Leave `USE_UPLOAD = False` to use a random image from the built-in test set.

In [ ]:
import os, zipfile, random
from pathlib import Path

USE_UPLOAD = False  # set True to upload your own images

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}

def _collect_images(paths):
    """Return all image files found under the given paths (files or directories)."""
    imgs = []
    for p in map(Path, paths):
        if p.is_dir():
            for ext in IMAGE_EXTS:
                imgs.extend(p.rglob(f'*{ext}'))
        elif p.suffix.lower() in IMAGE_EXTS:
            imgs.append(p)
    return sorted(set(imgs))

if USE_UPLOAD:
    from google.colab import files as _colab_files
    print('Select one image, multiple images (Ctrl/Cmd+click), or a ZIP of images:')
    uploaded = _colab_files.upload()

    upload_dir = Path('/tmp/bear_img_uploads')
    upload_dir.mkdir(parents=True, exist_ok=True)

    staged = []
    for fname, data in uploaded.items():
        dest = upload_dir / fname
        dest.write_bytes(data)
        if fname.lower().endswith('.zip'):
            extract_dir = upload_dir / Path(fname).stem
            extract_dir.mkdir(exist_ok=True)
            with zipfile.ZipFile(dest) as z:
                z.extractall(extract_dir)
            staged.append(extract_dir)
        else:
            staged.append(dest)

    SAMPLE_IMGS = [str(p) for p in _collect_images(staged)]
    if not SAMPLE_IMGS:
        raise ValueError('No image files found in the upload. Please upload .jpg/.png files or a ZIP containing them.')
    print(f'Found {len(SAMPLE_IMGS)} image(s)')
else:
    test_images = sorted(Path('Bear detection.v3i.yolov8-obb/test/images').glob('*.jpg'))
    SAMPLE_IMGS = [str(random.choice(test_images))]
    print(f'Sample image: {SAMPLE_IMGS[0]}')

SAMPLE_IMG = SAMPLE_IMGS[0]   # kept for the segmentation cell in Section 6

In [ ]:
from src.inference.detector import BearDetector
from IPython.display import display
from PIL import Image
import cv2

detector = BearDetector(
    weights_path   = BEST_PT,
    conf_threshold = cfg['inference']['detection_conf_threshold'],
    iou_threshold  = cfg['inference']['detection_iou_threshold'],
    device         = DEVICE,
)

for img_path in SAMPLE_IMGS:
    result = detector.predict_image(img_path, annotate=True)
    print(f'\n{Path(img_path).name}  —  {len(result["boxes"])} detection(s)')
    for i, (box, score, label) in enumerate(
            zip(result['boxes'], result['scores'], result['labels'])):
        print(f'  [{i}] {label}  conf={score:.3f}  box={[round(v,1) for v in box]}')
    annotated_rgb = cv2.cvtColor(result['annotated_image'], cv2.COLOR_BGR2RGB)
    display(Image.fromarray(annotated_rgb).resize((640, 480)))

## 5 · Detect + Track Bears in Videos

**Input options** (set `USE_UPLOAD = True` to activate the file picker):

| What you select | Result |
|-----------------|--------|
| One video (`.mp4`, `.avi`, `.mov`, …) | Processes that video |
| Multiple videos (Ctrl/Cmd+click) | Processes each video in sequence |
| A `.zip` of videos or folders of videos | Extracts and processes everything inside |

With `USE_UPLOAD = False` the notebook uses `data/sample/bear_sample.mp4` from the repository.

Each output video shows bears with **coloured bounding boxes and a track ID** that stays consistent across frames.

In [ ]:
import os, zipfile
from pathlib import Path

USE_UPLOAD = False  # set True to upload your own video(s)

VIDEO_EXTS        = {'.mp4', '.avi', '.mov', '.mkv', '.webm', '.m4v'}
SAMPLE_VIDEO_PATH = 'data/sample/bear_sample.mp4'

def _collect_videos(paths):
    """Return all video files found under the given paths (files or directories)."""
    vids = []
    for p in map(Path, paths):
        if p.is_dir():
            for ext in VIDEO_EXTS:
                vids.extend(p.rglob(f'*{ext}'))
        elif p.suffix.lower() in VIDEO_EXTS:
            vids.append(p)
    return sorted(set(vids))

if USE_UPLOAD:
    from google.colab import files as _colab_files
    print('Select one video, multiple videos (Ctrl/Cmd+click), or a ZIP of videos:')
    uploaded = _colab_files.upload()

    upload_dir = Path('/tmp/bear_vid_uploads')
    upload_dir.mkdir(parents=True, exist_ok=True)

    staged = []
    for fname, data in uploaded.items():
        dest = upload_dir / fname
        dest.write_bytes(data)
        if fname.lower().endswith('.zip'):
            extract_dir = upload_dir / Path(fname).stem
            extract_dir.mkdir(exist_ok=True)
            with zipfile.ZipFile(dest) as z:
                z.extractall(extract_dir)
            staged.append(extract_dir)
        else:
            staged.append(dest)

    INPUT_VIDEOS = [str(p) for p in _collect_videos(staged)]
    if not INPUT_VIDEOS:
        raise ValueError('No video files found in the upload. Please upload .mp4/.avi files or a ZIP containing them.')
    print(f'Found {len(INPUT_VIDEOS)} video(s)')

else:
    if not os.path.isfile(SAMPLE_VIDEO_PATH):
        raise FileNotFoundError(
            f'Sample video not found: {SAMPLE_VIDEO_PATH}\n'
            'Add a video file at that path in the repository, or set USE_UPLOAD = True.'
        )
    INPUT_VIDEOS = [SAMPLE_VIDEO_PATH]
    print(f'Using sample video: {SAMPLE_VIDEO_PATH}')

INPUT_VIDEO = INPUT_VIDEOS[0]   # kept for the segmentation cell in Section 6
print(f'\n{len(INPUT_VIDEOS)} video(s) queued for processing')

In [ ]:
from src.tracking.sort_tracker import SORTTracker
from pathlib import Path

Path('outputs/videos').mkdir(parents=True, exist_ok=True)
OUTPUT_VIDEOS = []

for vid_path in INPUT_VIDEOS:
    stem     = Path(vid_path).stem
    out_path = f'outputs/videos/{stem}_tracked.mp4'

    tracker_sort = SORTTracker(
        max_age       = cfg['tracking']['max_age'],
        min_hits      = cfg['tracking']['min_hits'],
        iou_threshold = cfg['tracking']['iou_threshold'],
    )
    summary = detector.process_video(
        video_path  = vid_path,
        output_path = out_path,
        tracker     = tracker_sort,
    )
    print(f'{Path(vid_path).name}: {summary}')
    OUTPUT_VIDEOS.append(out_path)

OUTPUT_VIDEO = OUTPUT_VIDEOS[0]   # kept for the segmentation cell in Section 6

In [ ]:
import os
from google.colab import files

for out_vid in OUTPUT_VIDEOS:
    h264_path = out_vid.replace('.mp4', '_h264.mp4')
    ret = os.system(f'ffmpeg -y -i "{out_vid}" -vcodec libx264 -crf 23 "{h264_path}" -loglevel error')
    download_path = h264_path if ret == 0 else out_vid
    print(f'Downloading: {download_path}')
    files.download(download_path)

## 6 · Instance Segmentation

**What is instance segmentation?**
Instead of just drawing a rectangle around a bear, the model draws a precise pixel-level mask — it colours exactly the pixels that belong to each bear.

**Why aren't we training our own segmentation model here?**
Training a segmentation model requires images where every bear is outlined with a polygon (not just a box). The dataset in this repo only has bounding-box labels, so we can't train from scratch.

**What this section does instead:**
We load YOLOv8n-seg weights that were pre-trained on the COCO dataset. COCO contains a generic "bear" class (class ID 21), so the model can already detect and segment bears — just not bear *faces* specifically. We filter predictions to class 21 so only bears are shown.

> **To train your own bear segmentation model:** annotate your images with polygon masks
> (e.g. using [Roboflow](https://roboflow.com) or [CVAT](https://cvat.ai) in YOLO-seg format),
> then uncomment the training block below and point it at your dataset.

In [ ]:
from src.segmentation.infer_segmentation import BearSegmentor

# yolov8n-seg.pt = COCO pretrained segmentation model (downloads automatically, ~6 MB).
# COCO class 21 = "bear", so this works on whole-body bears out of the box.
# Replace 'yolov8n-seg.pt' with your own trained weights to get bear-face masks.
segmentor = BearSegmentor(
    weights_path   = 'yolov8n-seg.pt',
    conf_threshold = 0.25,
    device         = DEVICE,
    classes        = [21],   # restrict to COCO class 21 = "bear"
)

seg_result = segmentor.predict_image(SAMPLE_IMG, annotate=True)

print(f'Instances : {len(seg_result["boxes"])}')
print(f'Masks     : {len(seg_result["masks"])}')
for label, score in zip(seg_result['labels'], seg_result['scores']):
    print(f'  {label}  conf={score:.3f}')

seg_rgb = cv2.cvtColor(seg_result['annotated_image'], cv2.COLOR_BGR2RGB)
display(Image.fromarray(seg_rgb).resize((640, 480)))

### Run segmentation on a video

The same `INPUT_VIDEO` from Section 5 is used. Each frame gets instance masks overlaid on detected bears.
Re-run the video upload cell above to change the source video.

In [ ]:
# ── Train your own bear segmentation model ──────────────────────────────────────
# 1. Annotate your images with polygon masks in YOLO-seg format
#    (Roboflow and CVAT both export this format directly)
# 2. Set the path below to your dataset folder (must contain data.yaml,
#    train/images/, train/labels/, valid/images/, valid/labels/)
# 3. Uncomment and run

# cfg['data']['segmentation']['data_dir'] = 'path/to/your/seg-dataset'
# cfg['training']['segmentation']['epochs'] = 50

# from src.segmentation.train_segmentation import SegmentationTrainer
# seg_trainer = SegmentationTrainer(cfg)
# seg_output  = seg_trainer.train()
# print('Best weights:', seg_output['best_weights'])

print('Training block is ready — annotate your data and uncomment the lines above.')

In [ ]:
import os, cv2
from pathlib import Path

SEG_OUTPUT_VIDEO = 'outputs/videos/bear_segmented.mp4'
Path(SEG_OUTPUT_VIDEO).parent.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(INPUT_VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS) or 10.0
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(SEG_OUTPUT_VIDEO, fourcc, fps, (w, h))

frame_count = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    result = segmentor.predict_image(frame, annotate=True)
    out.write(result['annotated_image'])
    frame_count += 1

cap.release()
out.release()
print(f'Processed {frame_count} frames → {SEG_OUTPUT_VIDEO}')

# Re-encode to H.264 for browser playback
SEG_H264 = SEG_OUTPUT_VIDEO.replace('.mp4', '_h264.mp4')
ret_code  = os.system(f'ffmpeg -y -i "{SEG_OUTPUT_VIDEO}" -vcodec libx264 -crf 23 "{SEG_H264}" -loglevel error')
if ret_code == 0:
    print(f'Re-encoded to H.264: {SEG_H264}')
    from google.colab import files
    files.download(SEG_H264)
else:
    from google.colab import files
    files.download(SEG_OUTPUT_VIDEO)

## 7 · Evaluation

In [ ]:
# Validate detection model on the test split
from src.training.train_detection import DetectionTrainer
import json

det_trainer = DetectionTrainer(cfg)
metrics = det_trainer.validate(BEST_PT)

print('\n=== Detection Metrics ===')
for k, v in metrics.items():
    print(f'  {k:20s}: {v}')

In [ ]:
# View the PR curve saved during training
from pathlib import Path
from IPython.display import display
from PIL import Image
import glob

pr_files = list(Path('outputs/metrics').glob('*pr_curve*.png')) + \
           list(Path('outputs/metrics').glob('*precision_recall*.png'))

if pr_files:
    display(Image.open(pr_files[0]))
else:
    print('No PR curve found yet — run evaluate_detection() first.')

## 8 · MLflow Experiment Tracking

**What is MLflow?**
MLflow records every training run — hyperparameters, per-epoch metrics, and model weights — so you can compare experiments and reproduce results.

**How it works in this project:**
Ultralytics automatically logs every training run to `outputs/experiments/mlruns/` via its built-in MLflow callback. No extra configuration needed — just train and the data is there.

**Two ways to inspect results:**

| Method | How |
|--------|-----|
| JSON summary | Run the first block below — one `.json` file per run in `outputs/experiments/` |
| MLflow UI | Uncomment the ngrok block below — opens a browser UI to compare runs side-by-side |

> **Why ngrok?** Colab runs on a remote machine. `127.0.0.1:5000` is that machine's localhost, not yours. Ngrok creates a public URL that tunnels to it.

In [ ]:
import json
from pathlib import Path

# ── Option 1: View logged metrics from JSON ──────────────────────────────────────
log_files = sorted(Path('outputs/experiments').glob('*.json'))
if log_files:
    print('=== Experiment JSON logs ===')
    for f in log_files:
        print(f'\n--- {f.name} ---')
        with open(f) as fh:
            print(json.dumps(json.load(fh), indent=2))
else:
    print('No experiment logs yet — run training first (Section 3).')

# ── Option 2: Launch MLflow UI via ngrok ────────────────────────────────────────
# Ultralytics automatically logs every training run to outputs/experiments/mlruns/.
# The MLflow server runs on Colab's localhost, which your browser cannot reach
# directly — ngrok creates a public tunnel so you can open the UI.
#
# Uncomment and run this block after at least one training run:
#
# !pip install -q pyngrok
# !pkill -f "mlflow server" 2>/dev/null; sleep 1   # kill any previous server
# !mlflow server \
#     --backend-store-uri outputs/experiments/mlruns \
#     --host 127.0.0.1 --port 5000 \
#     --workers 1 &
# import time; time.sleep(3)   # wait for server to start
# from pyngrok import ngrok
# ngrok.kill()                  # close any existing tunnels
# public_url = ngrok.connect(5000)
# print(f'✅ MLflow UI → {public_url}')
# print('Open the link above in your browser to compare runs.')

## Tips for Colab

| Tip | How |
|-----|-----|
| Save your trained weights | `from google.colab import files; files.download(BEST_PT)` |
| Use Google Drive | `from google.colab import drive; drive.mount('/content/drive')` |
| Monitor GPU usage | `!nvidia-smi` |
| Check YOLO training live | Open `outputs/models/detection/*/results.csv` |
| Resume training | Set `pretrained_weights` in config to a previous `.pt` file |

---

**GitHub:** https://github.com/danort92/Bear-Detector  
**Author:** Danilo Ortelli
